In [2]:
import numpy as np

In [4]:
## [Для всех] Чтение кода — без запуска
#Сначала напишите ответ и «почему», потом запустите.
#```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

# Мои предсказания и объяснения:
#1. 5 (2,3) -  индекс a[1, 2] = 5, размерность a.shape = (2, 3)
#2. 1 8 - индех макс значения 1 и само максим значение 8, тк 8 две индекс вернет первый макс зн
#3. 99 -  тк элемент b[0] был изменен на 99, соответственно эл a[0] изменен на 99
#4. 2 0.5 - a>3 дает [False, True, True, False], sum() = 2, mean() = 2/4
#5. (2, 3) 0.25 - вывод размерность матрцы из нулей и вывод второго элемента массива

5 (2, 3)
1 8
99
2 0.5
(2, 3) 0.25


In [5]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [39]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
   rows = [
        r for r in csv.DictReader(f)
        if "test" not in r["customer_id"].lower()
    ]

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5460,) float64 bool


In [41]:
#1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
max_order = amount.argmax()
print(f"1. Самый дорогой заказ (индекс {max_order}): сумма = {amount[max_order]}, клиент = {cust[max_order]}, город = {city[max_order]}")

1. Самый дорогой заказ (индекс 3314): сумма = 139040.0, клиент = C02122, город = Шымкент


In [42]:
#2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
cost_paid = amount[paid].sum()
cost_unpaid = amount[~paid].sum()
print(f"2. Выручка оплаченных заказов = {cost_paid}, неоплаченных = {cost_unpaid}")

2. Выручка оплаченных заказов = 71909390.0, неоплаченных = 7974000.0


In [43]:
#3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?
shymkent_mask = (city == "Шымкент")
shymkent_count = shymkent_mask.sum()
shymkent_share = shymkent_mask.mean() * 100
print(f"3. Заказов в Шымкенте: {shymkent_count} ({shymkent_share:.2f}% от всех заказов)")

3. Заказов в Шымкенте: 804 (14.73% от всех заказов)


In [51]:
#4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
cond_all = paid & (amount > 50000)
loss_all = (amount[cond_all] * 0.15).sum()

print(f" 4. Недополученная прибыль : {loss_all}")

 4. Недополученная прибыль : 1202427.0


In [71]:
#5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
print("\n5. Доля быстрой доставки (1–2 дня) по городам:")
for c in np.unique(city):
    # Берем заказы конкретного города, где указан срок доставки (не NaN)
    valid_city = (city == c) & ~np.isnan(days)
    fast_delivery = valid_city & (days >= 1) & (days <= 2)

    share = (fast_delivery.sum() / valid_city.sum()) * 100
    print(f"   {c}: {share:.1f}%")


5. Доля быстрой доставки (1–2 дня) по городам:
   Актобе: 62.8%
   Алматы: 66.5%
   Астана: 65.4%
   Атырау: 66.7%
   Караганда: 71.0%
   Шымкент: 30.7%


In [70]:
#6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
#```python
#avg = amount.mean()
#print("Средний чек:", round(avg))
#```
clean_paid_mask = paid
avg_check = amount[clean_paid_mask].mean()
print(f"Средний чек: {round(avg_check)}")

Средний чек: 14672


In [ ]:
## [Челлендж]
#7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
#8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
#  (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
#  Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)